In [1]:
import cobra

import os
import multiprocessing
import gc
import warnings
from tqdm import tqdm
import ast
import copy
import time

import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np


import sys
sys.path.insert(1, '../../scripts/') # comment out in python script
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    from utils.load_environmental_variables import build_files_path, processed_data_path
    from utils import parameters as params
    from utils import machinery as mach
    
    from utils import functions as func  
    from preprocess import parse_complex
    
    import core
    from core.reaction import Expression_Reaction, Metabolic_Reaction, Complex_Degradation_Reaction, Protein_Degradation_Reaction, to_metabolic_reaction
    from core.model import ME_Model
    
    with func.HiddenPrints():
        from macromolecules.macromolecule import Macromolecule
        from macromolecules.protein import Protein
        from macromolecules.complex import Complex, Ribosomal_Complex
        
        import expression.build_mrna_expression_reactions as build_mrna
        from expression import gene_information
        from expression.protein_expression import ubiquitin, degradation
        from expression.protein_expression import build_protein_expression_reactions as build_protein
        
        from uniform_processes.build_ribosome_biogenesis_reactions import build_ribosome
        from uniform_processes.build_trna_expression_reactions import trna_biogenesis_reactions
        from uniform_processes import biomass

No objective coefficients in model. Unclear what should be optimized


# Generate Protein Expression Reactions for All Machinery

In [2]:
def get_all_expression_reactions(hgnc_id, reactions, ub_args, psim = params.psim_me, machinery_list = mach.metabolic_machinery, 
                             compress_mrna = False, nonmachinery_locations = list()):
    '''Generates all the expression reactions for a given protein from the HGNC ID and the PSIM'''
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        with func.HiddenPrints():
            gene_info = gene_information.generate(hgnc_id, psim, machinery_list, reactions = reactions, nonmachinery_locations = nonmachinery_locations)
            mrna_reactions, mrna_transcript_c, mrna_deg_proxy  = build_mrna.get_mrna_expression_reactions(gene_info, compress_mrna = compress_mrna)
            protein_reactions, protein_metabolites = build_protein.get_protein_expression_reactions(gene_info, mrna_transcript_c, mrna_deg_proxy, 
                                                                                                    ub_args = ub_args)
            
    return mrna_reactions + protein_reactions, protein_metabolites

def get_expression_machinery(reactions):
    gene_reaction_map = func.create_gene_reaction_map(reactions)
    expression_machinery_me = list(gene_reaction_map)
    if 'ribosome' in expression_machinery_me:
        expression_machinery_me.remove('ribosome')
    return gene_reaction_map, expression_machinery_me

def parse_complex_degradation_reaction_id(r_id):
    '''Generates universal reaction ID analagous to func.parse_me_reaction_id specifically for Complex_Degradation_Reaction'''
    if r_id.count('_COMPLEX_') == 1:
        return r_id[r_id.index('_COMPLEX_') + len('_COMPLEX_'):]
    else:
        return '_'.join(r_id.split('_')[[i for i in range(len(r_id.split('_'))) if r_id.split('_')[i] == 'COMPLEX'][-1]+1:])

def get_ko(mach, knock_out):
    '''Determine whether a machinery list intersects with a knock_out genes list'''
    if len(knock_out) == 0 or len(set(mach).intersection(knock_out))==0:
        return False
    else:
        return True

def get_complex_df(reactions, knock_out):
    '''Generate the complex df
    
    Paramaters
    ----------
    reactions: list
        each element is a cobra.core.Reaction object
    knock_out: list
        each element is a string representing a gene expressed in the model which should be knocked out
    reaction_category: str
        either a "metabolic_"
    '''
    
    complex_df = pd.DataFrame(columns = ['reaction_id', 'compartment', 'machinery', 'is_complex', 'creates_multiple_reactions', 
                                            'knock_out'])

    for r in tqdm(reactions):
        compartment_ = func.get_reaction_compartment(r)

        ko = True
        #YOU ARE HERE
        if len(r.genes) == 1:
            ko = get_ko(mach = [list(r.genes)[0].id], knock_out = knock_out)
            complex_df.loc[complex_df.shape[0], :] = [r.id, compartment_, list(r.genes)[0].id, False, False, ko]
        elif 'and' in r.gene_reaction_rule and 'or' in r.gene_reaction_rule: 
            machinery_final = parse_complex.eval_complex(r.gene_reaction_rule)
            for m in machinery_final:
                if type(m) == list:
                    ko = get_ko(mach = m, knock_out = knock_out)
                    complex_df.loc[complex_df.shape[0], :] = [r.id, compartment_, ';'.join(m), True, True, ko]
                else:
                    ko = get_ko(mach = [m], knock_out = knock_out)
                    complex_df.loc[complex_df.shape[0], :] = [r.id, compartment_, m, False, True, ko]
        elif 'or' in r.gene_reaction_rule:
            machinery = [g.id for g in list(r.genes)]
            for m in machinery:
                ko = get_ko(mach = [m], knock_out = knock_out)
                complex_df.loc[complex_df.shape[0], :] = [r.id, compartment_, m, False, True, ko]
        elif 'and' in r.gene_reaction_rule:
            m = sorted([g.id for g in r.genes])
            ko = get_ko(mach = m, knock_out = knock_out)
            complex_df.loc[complex_df.shape[0], :] = [r.id, compartment_, ';'.join(m), True, False, ko]
        else: # no genes
            pass

    return complex_df



In [3]:
def emm_par(hgnc_id, gene_reaction_map, ub_args, compress_mrna, non_machinery):
    # None bc will add later for expression model specific to this
    nml = list()
    if hgnc_id in non_machinery:
        nml = non_machinery[hgnc_id]
    expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, 
                                        reactions = gene_reaction_map[hgnc_id], compress_mrna = compress_mrna, 
                                            ub_args = ub_args, nonmachinery_locations = nml)
    id_protein_map = {p.compartment: p for p in protein_metabolites} # store compartments and metabolite objects for each gene
    return id_protein_map, expr_reactions

In [4]:
class me_builder():
    def __init__(self, n_cores = os.cpu_count(), compress_mrna = False, dummy_protein = True, 
                 deg_args = {'couple': True, 'reversible_complex_formation': False, 'nonenzyme_degradation': False, 
                          'complex_degradation': True}, check_all = True, knock_out = None, non_machinery = dict(),
                psim_me = params.psim_me, m_model = params.human_model):
        
        '''Generates a human ME_model. 
    
        Parameters
        ----------
        n_cores: int
            # of cores to parallelize on; defaults to all available cores
        minimal_proteome: bool
            For reactions with OR in the GPR, the builder by default (False) generates a 
            separate reaction for each protein complex (False). If True, builder instead will create one reaction, 
            choosing the protein complex with the lowest molecular weight to catalyze the reaction. If a reaction
            has multiple enzyme options with the same molecular weight, will randomly choose one. Will not consider
            a complex that contains a knocked out gene. 
        compress_mrna: bool
            If true, will merge the 3 linear mrna reactions--transcription, processing, and nuclear export--for each
            gene into a single reaction
        dummy_protein: bool [True]
            whether to add a representative dummy protein to catalyze orphan reactions 
        deg_args: dict
            A number of options related to protein and complex degradation. Becomes important in slow growth conditions.
            Note the default values focus on coupling fluxes and degrading the specific enzymes associated with 
            each reaction. 

            Key value pairs:
                "couple": bool
                    Whether to explicitly couple enzyme degradation reactions to metabolic catalysis. Becomes 
                    particularly important in slow growth conditions.
                "reversible_complex_formation": bool
                    Whether reactions to form complexes are reversible (<->) or not (-->). Setting to True may make
                    model more efficient (reuse of proteins involved in catalysis of multiple reactions in same compartment)
                "nonenzyme_degradation": bool
                    Whether to retain degradation reactions (associated with the build_protein_expression script) for
                    proteins that form complexes rather than become monomeric enzymes; i.e., all individual complex 
                    subunits have their own protein degradation reaction. Note that even if set to False, 
                    protein intermediates associated with the monomeric enzyme that had degradation rections are retained.
                    Regardless of this parameter, only the specific enzymatic degradation reaction associated with the 
                    catalysis reaction will be coupled. Independent of complex_degradation and 
                    reversible_complex_formation arguments.
                "complex_degration": bool
                    Whether to generate degradation reactions for whole complexes in addition to individual monomers
                    (required for coupling)
        check_all: bool
            Whether to check that building is proceeding correctly. Increases run time
        model_id: str
            id for the me model
        knock_out: list
            each element is a string representing a gene expressed in the model which should be knocked out
            
            *Note: you may want to knock-out during building if setting minimal_proteome = True and knocking out a 
            gene that participates in a OR GPR rule(in case it is the one that is selected); otherwise 
            me_model.knock_out() method should suffice
        non_machinery: dictionary
            keys are HGNC IDs, values are a list of strings, each element of which represents a compartment
            within the metabolic model for the gene to be expressed. 
            Exceptions are ubiquitin genes (HGNC:12468, HGNC:12463) and ribosomal genes
        '''

        # check deg args
        if not deg_args['complex_degradation']:
            if not deg_args['nonenzyme_degradation'] or not deg_args['reversible_complex_formation']:
                err = 'If complex degradation is not included, the formation must be reversible and the individual '
                err += 'components must be able to be degraded. Set nonenzyme_degradation and reversible_complex_formation'
                err += ' to True or complex_degradation to True'
                raise ValueError(err)
        if deg_args['couple'] and not deg_args['complex_degradation']:
            raise ValueError('In order to couple metabolic catalysis to enzyme degradation, complex_degradation must be True')
        self.deg_args = deg_args
        
        for exception in ['HGNC:12468', 'HGNC:12463'] + mach.rl['HGNC ID (gene)'].tolist() + mach.rs['HGNC ID (gene)'].tolist():
            if exception in non_machinery:
                warnings.warn(exception + ' is a ribosomal or ubiquitin-related gene, which cannot be specified as non-machinery, removing')
                del non_machinery[exception]
        self.non_machinery = non_machinery
        
        
        if knock_out is None:
            self.knock_out = list()
        else:
            self.knock_out = knock_out 
        if len(set(self.knock_out).intersection(mach.expression_machinery))>0:
            raise ValueError('Knock outs can only be applied to metabolic machinery and non-machinery, not expression machinery')
        if len(set(self.knock_out).intersection(self.non_machinery)) > 0:
            raise ValueError('Speficied knocking out of genes that are also specified to be expressed as non-machinery')
            
        self.psim_me = psim_me
        self.m_model = m_model
        
        self.n_cores = n_cores
        if self.n_cores in [0,1,None]:
            self._par = False
        else:
            self._par = True
        
        # get pre-generated reactions - the compress_mrna arg requires that they be run with that input
        self.compress_mrna = compress_mrna
        print('Generate ubiquitin reactions for proteasomal degradation')
        self.ub_args = ubiquitin.express_ubiquitin(compress_mrna = self.compress_mrna)
        print('Generate ribosome')
        ribosomal_reactions, self.ribosome_complex_c = build_ribosome(self.ub_args, self.compress_mrna, 
                                                        self.deg_args['reversible_complex_formation'])
        
        self.dummy_protein = dummy_protein
            
        
        self.deorphaned = None
        self.orphan = None
        
        self.me_reactions = trna_biogenesis_reactions + ribosomal_reactions + self.ub_args['ub_reactions']
        # map HGNC ID to a dictionary of compartments and cobra.Metabolite proteins
        self.id_protein_map = dict() 
        self.complex_id_metabolite_map = dict() # map complex id to the complex cobra.Metabolite
        
        self._ko_id_protein_map = dict() 
        self._ko_complex_id_metabolite_map = dict() 
        
        self.id_reactions_map = dict()
        self.complex_reactions_map = dict()

        self.check_all = check_all
    
    def express_metabolic_enzymes(self):
        '''Get protein expression reactions for all metabolic enzymes and user-input non-machinery'''

        # get protein expression for all metabolic reactions
        print('Generate protein expression reactions for metabolic enzymes and non-machinery')


        self.loop_machinery = list(set(mach.metabolic_machinery + list(self.non_machinery)))

        gene_reaction_map = func.create_gene_reaction_map(params.human_model.reactions)
        for hgnc_id in self.non_machinery:
            if hgnc_id not in gene_reaction_map:
                gene_reaction_map[hgnc_id] = None


        iterable = set(self.loop_machinery).difference(self.knock_out)
        if not self._par:
            for hgnc_id in tqdm(iterable):
                nml = list()
                if hgnc_id in self.non_machinery:
                    nml = self.non_machinery[hgnc_id]
                expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, 
                                                        reactions = gene_reaction_map[hgnc_id],
                                                      compress_mrna = self.compress_mrna, 
                                                        ub_args = self.ub_args, nonmachinery_locations = nml)
                self.id_protein_map[hgnc_id] = {p.compartment: p for p in protein_metabolites} # store compartments and metabolite objects for each gene
                self.id_reactions_map[hgnc_id] = expr_reactions
                self.me_reactions += expr_reactions
        else:
            pool = multiprocessing.Pool(processes = self.n_cores)
            try:
                n_iter = len(iterable)
                args = zip(iterable, [gene_reaction_map]*n_iter, [self.ub_args]*n_iter, [self.compress_mrna]*n_iter, [self.non_machinery]*n_iter)
                mm = pool.starmap(emm_par, args)
                pool.close()
                pool.join()
                gc.collect()
            except:
                pool.close()
                pool.join()
                gc.collect()
                raise ValueError('Parallelization failed')
            self.id_protein_map = dict(zip(iterable, [i[0] for i in mm]))
            expr_reactions = [i[1] for i in mm]
            self.id_reactions_map = dict(zip(iterable, expr_reactions))
            self.me_reactions += func.flatten_list(expr_reactions)
            del expr_reactions

        for hgnc_id in self.knock_out:
            # None bc will add later for expression model specific to this
            expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, reactions = gene_reaction_map[hgnc_id], 
                                                                               compress_mrna = self.compress_mrna, 
                                                                              ub_args = self.ub_args)
            self._ko_id_protein_map[hgnc_id] = {p.compartment: p for p in protein_metabolites} # store compartments and metabolite objects for each gene

    def express_expression_enzymes(self):
        
        #This method continues to add any expression module machinery that may have arisen from adding expression 
        #reactions for expression machinery. 
        
        gene_reaction_map, expression_machinery_me = get_expression_machinery(self.me_reactions)

        for hgnc_id in tqdm(list(set(expression_machinery_me).difference(self.knock_out))):
            nml = list()
            if hgnc_id in self.non_machinery:
                nml = self.non_machinery[hgnc_id]
            expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, machinery_list = expression_machinery_me,
                                                  reactions = gene_reaction_map[hgnc_id], compress_mrna = self.compress_mrna, 
                                                  ub_args = self.ub_args, nonmachinery_locations = nml)


            if hgnc_id not in set(expression_machinery_me).intersection(self.loop_machinery):
                if hgnc_id in self.id_protein_map.keys():
                    raise ValueError('Some genes not accounted for when generating metabolic machinery expression reactions')
                else:
                    self.id_protein_map[hgnc_id] = {p.compartment:p for p in protein_metabolites}
            # when there is machinery overlap between metabolic and expression module, deal with compartment overlap 
            else:
                ids_to_keep = list(set([r.id for r in expr_reactions]).difference([r.id for r in self.me_reactions]))
                expr_reactions = [r for r in expr_reactions if r.id in ids_to_keep]

                temp_map = {p.compartment:p for p in protein_metabolites}
                for comp, met in temp_map.items():
                    if comp not in self.id_protein_map[hgnc_id]: 
                        self.id_protein_map[hgnc_id][comp] = met
                    elif not met.non_machinery: # in the case that it was specified as non-machinery during metabolic iteration
                        met = self.id_protein_map[hgnc_id][comp]
                        met.non_machinery = False 

            if hgnc_id not in self.id_reactions_map.keys():
                self.id_reactions_map[hgnc_id] = expr_reactions
            else:
                self.id_reactions_map[hgnc_id] += expr_reactions

            self.me_reactions += expr_reactions

        gene_reaction_map_2, expression_machinery_me_2 = get_expression_machinery(self.me_reactions)
        new_expression_machinery = list(set(expression_machinery_me_2).difference(expression_machinery_me + self.knock_out))

        counter = 1
        while len(new_expression_machinery)>0:  # this condition leaves possibility that an existing machinery but with a new compartment is added and not accounted for
            print('No. iterations for new expression machinery: {}'.format(counter))
            nml = list()
            if hgnc_id in self.non_machinery:
                nml = self.non_machinery[hgnc_id]
            for hgnc_id in tqdm(list(set(expression_machinery_me_2))):
                expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, machinery_list = expression_machinery_me_2,
                                                      reactions = gene_reaction_map_2[hgnc_id], compress_mrna = self.compress_mrna, 
                                                      ub_args = self.ub_args, nonmachinery_locations = nml)


                if hgnc_id not in set(expression_machinery_me_2).intersection(expression_machinery_me + mach.metabolic_machinery):
                    if hgnc_id in self.id_protein_map.keys():
                        raise ValueError('Some genes not accounted for when generating metabolic machinery expression reactions')
                    else:
                        self.id_protein_map[hgnc_id] = {p.compartment:p for p in protein_metabolites}

                # when there is machinery overlap between metabolic and expression module, deal with compartment overlap 
                else:
                    ids_to_keep = list(set([r.id for r in expr_reactions]).difference([r.id for r in self.me_reactions]))
                    expr_reactions = [r for r in expr_reactions if r.id in ids_to_keep]

                    temp_map = {p.compartment:p for p in protein_metabolites}
                    for comp, met in temp_map.items():
                        if comp not in self.id_protein_map[hgnc_id]: 
                            self.id_protein_map[hgnc_id][comp] = met
                        elif not met.non_machinery: # in the case that it was specified as non-machinery during metabolic iteration
                            met = self.id_protein_map[hgnc_id][comp]
                            met.non_machinery = False 

                if hgnc_id not in self.id_reactions_map.keys():
                    self.id_reactions_map[hgnc_id] = expr_reactions
                else:
                    self.id_reactions_map[hgnc_id] += expr_reactions

                self.me_reactions += expr_reactions

            # get protein expression reactions for all expression module reactions
            expression_machinery_me = copy.deepcopy(expression_machinery_me_2)
            gene_reaction_map_2, expression_machinery_me_2 = get_expression_machinery(self.me_reactions)
            new_expression_machinery = list(set(expression_machinery_me_2).difference(expression_machinery_me + self.knock_out))
            counter += 1
        
        self._clean_non_machinery()
    
    def _clean_non_machinery(self):
        ipm = self.id_protein_map.copy()
        for k,v in self._ko_id_protein_map.items():
            ipm[k] = v
        nm2 = self.non_machinery.copy()
        for hgnc_id, compartments in nm2.items():
            for compartment in self.non_machinery[hgnc_id]:
                if not ipm[hgnc_id][compartment].non_machinery: # when overlaps with enzyme
                    self.non_machinery[hgnc_id].remove(compartment)
            if len(self.non_machinery[hgnc_id]) == 0:
                del self.non_machinery[hgnc_id]
        del nm2, ipm 
        
    
    def express_dummy_protein(self):
        if self.dummy_protein:
            print('Express dummy protein')
            dummy_psim = func.average_protein_features(psim_me = self.psim_me, 
                                                      protein_ids = sorted(self.id_protein_map.keys()), 
                                                     context_specific = True)

            dummy_psim = func.average_protein_features(psim_me = self.psim_me, 
                                                  protein_ids = sorted(self.id_protein_map.keys()), 
                                                 context_specific = True)

            dummy_reactions, dm = get_all_expression_reactions(hgnc_id = 'HGNC:DUMMY', psim = dummy_psim, machinery_list = [], 
                                                                reactions = None, compress_mrna = self.compress_mrna, 
                                                              ub_args = self.ub_args, nonmachinery_locations = ['c']) 
            dm[0].non_machinery = False
            for r in dummy_reactions:
                for m in r.metabolites:
                    if isinstance(m, Protein) and m.id.startswith('HGNC:DUMMY'): # str requirement to avoid converting ub proteins
                        m.dummy = True

            self.dummy_protein = {'protein_metabolite': dm[0], 'dummy_expression_reactions': dummy_reactions}

            srs = [sr for sr in list(self.dummy_protein['protein_metabolite'].reactions)if self.dummy_protein['protein_metabolite'] in sr.products and \
                   not isinstance(sr, Protein_Degradation_Reaction)]
            if len(srs) != 1:
                raise ValueError(self.dummy_protein['protein_metabolite'].id + ' has an incorrect number of associated synthesis reactions')
            srs[0].synthesis, srs[0].synthesis_type = True, 'protein'

            self.me_reactions += self.dummy_protein['dummy_expression_reactions']

        else:
            self.dummy_protein = None
            
    def get_complex_info(self):
        ######------------Metabolic Complexes
        print('Get metabolic module complex information')
        complex_df = get_complex_df(reactions = self.m_model.reactions, knock_out = self.knock_out)
        complex_df['category'] = 'metabolic_reaction'


        ######------------Expression Complexes
        print('Get expression module complex information')
        me_complex_df = get_complex_df(reactions = self.me_reactions, knock_out = self.knock_out)
        # deal with most expression reactions having redundant machinery in a concise manner:
        me_complex_df.reaction_id = me_complex_df.reaction_id.apply(lambda x: func.parse_me_reaction_id(x))
        me_complex_df.drop_duplicates(keep = 'first', inplace = True) # if all but reaction id HGNC were the same
        me_complex_df.reset_index(inplace = True, drop = True)
        me_complex_df['category'] = 'expression_reaction'

        ######-------------Merge Modules
        # merge bc will deal with duplicate complexes, in case there is duplicates b/w metabolic and expression module
        complex_df = pd.concat([complex_df, me_complex_df], axis = 0)
        complex_df.reset_index(inplace = True, drop = True)        
        
        print('Assign unique complex ids for unique machinery-compartment sets across all reactions')
        # assign complex ids for reactions that have complexes in them
        complex_df['complex_id'] = float('nan')
        complex_df.loc[complex_df[complex_df.is_complex].index, 'complex_id'] = complex_df.loc[complex_df[complex_df.is_complex].index, 'reaction_id']
        
        # if a reaction generates multiple complexes, make sure each complex has a unique ID
        crm_ = complex_df[(complex_df.creates_multiple_reactions) & (complex_df.is_complex)].reaction_id.unique()

        for crm in crm_:
            df = complex_df[(complex_df.reaction_id == crm) & (complex_df.is_complex)]
            if df.shape[0]>1: # reaction creates multiple complexes
                counter = 0
                for i in df.index:
                    complex_df.loc[i, 'complex_id'] = complex_df.loc[i, 'complex_id'] + '_' + str(counter)
                    counter += 1
        
        # if complexes are duplicated across different reactions assigned to the same compartment, 
        # generate a singular unique id
        dup_complexes = complex_df[complex_df.is_complex].duplicated(subset = ['compartment', 'machinery'], keep = 'first')
        dup_complexes = complex_df.loc[dup_complexes.index[np.where(dup_complexes)]]
        track_dups = dict()
        for i in dup_complexes.index:
            dups = complex_df[(complex_df.compartment == dup_complexes.loc[i,'compartment']) & (complex_df.machinery == dup_complexes.loc[i, 'machinery'])]
            new_id = '_'.join(dups.reaction_id)
            if new_id in track_dups.keys():
                track_dups[new_id] += 1
            else:
                track_dups[new_id] = 0

            new_id = new_id + '_' + str(track_dups[new_id])


            complex_df.loc[dups.index,'complex_id'] = new_id

        # those that didn't need _0
        simplify = [k for k,v in track_dups.items() if v == 0]
        for cid in simplify:
            complex_df.loc[complex_df[complex_df.complex_id == cid + '_0'].index, 'complex_id'] = complex_df[complex_df.complex_id == cid + '_0'].complex_id.apply(lambda x: x[:-2]).tolist()
        
        self.complex_df = complex_df    
        
    def generate_complex_reactions(self):
        # create a mapping of the unique self.complex_df ids to the actual complex metabolite
        unique_complexes = self.complex_df[self.complex_df.is_complex]
        unique_complexes = unique_complexes.drop_duplicates(subset = 'complex_id', keep = 'first')
        unique_complexes.reset_index(inplace = True, drop = True)

        self.complex_formation_reactions = list() # store all complex formation reactions
        complex_degradation_reactions = list()

        retain = list(set(func.flatten_list([i.split(';') for i in unique_complexes[~unique_complexes.knock_out.astype(bool)].machinery.tolist()])))
        self.additional_ko = list()

        counter = 0
        for i in unique_complexes.index:
            ko = unique_complexes.loc[i, 'knock_out']
            if ko:
                # get the  genes that are only expressed to participate as part of a complex that is knocked out
                self.additional_ko += list(set(unique_complexes.loc[i, 'machinery'].split(';')).difference(retain + self.knock_out))
            complex_id = unique_complexes.loc[i, 'complex_id']
            compartment = unique_complexes.loc[i, 'compartment']
            machinery = unique_complexes.loc[i, 'machinery'].split(';')
            machinery_metabolites = list()
            if not ko:
                counter_rib = 0
                for m in machinery:
                    if m != 'ribosome':
                        machinery_metabolites.append(self.id_protein_map[m][compartment])
                    else:
                        machinery_metabolites.append(self.ribosome_complex_c)
                        counter_rib += 1

                if counter_rib==0:
                    complex_metabolite = Complex(metabolites = machinery_metabolites, complex_id = complex_id)
                else:
                    complex_metabolite = Ribosomal_Complex(metabolites = machinery_metabolites, complex_id = complex_id)
                if len(complex_id) > 247: # ids that are too long
                    complex_metabolite.update_id(new_id = str(counter)) # complex_metabolite.udate_id()
                    counter += 1

                    new_id = complex_metabolite.id
                    self.complex_df.complex_id.replace(to_replace = complex_id, value = complex_metabolite.temp_id, 
                                                       inplace = True)

                complex_formation_reaction = complex_metabolite.form_complex(reversible = self.deg_args['reversible_complex_formation'], 
                                                                            synthesis = True, synthesis_type = 'complex')
                complex_formation_reaction.synthesis = True # for ribosomal complex formation
                if self.deg_args['complex_degradation']:
                    if complex_metabolite.compartment in ['c', 'n', 'r','g', 'pm']:
                        complex_degradation_reaction = degradation.degrade(complex_metabolite, **{'ub_args': self.ub_args})
                    elif complex_metabolite.compartment == 'e':
                        complex_degradation_reaction = list()
                    else:
                        complex_degradation_reaction = degradation.degrade(complex_metabolite)
#                     if counter_rib != 0: # ribosome, manually set attribute
#                         for rcdr in complex_degradation_reaction:
#                             r.ribosome_biogenesis = True
                    complex_degradation_reactions += complex_degradation_reaction
                else:
                    complex_degradation_reaction = list()

                self.complex_formation_reactions.append(complex_formation_reaction)
                self.complex_id_metabolite_map[complex_metabolite.temp_id] = complex_metabolite
                self.complex_reactions_map[complex_metabolite.temp_id] = [r.id for r in [complex_formation_reaction] \
                                                                          + complex_degradation_reaction]
            else:
                for m in machinery:
                    if m in self.id_protein_map:
                        machinery_metabolites.append(self.id_protein_map[m][compartment])
                    else:
                        machinery_metabolites.append(self._ko_id_protein_map[m][compartment])
                complex_metabolite = Complex(metabolites = machinery_metabolites, complex_id = complex_id)
                self._ko_complex_id_metabolite_map[complex_metabolite.temp_id] = complex_metabolite

        if self.deg_args['complex_degradation'] and self.check_all:
            # --check that machinery is the same for complex degradation (with exception of proteasomal degradation of ribosome)
            # as protein degradation

            # double check that only ribosomal degradation has additional machinery
            # this following code can be commented out if don't want to double check
            # keep, so in future iterations, can be used to iteratively add new machinery

            # this is all pretty hard-coded, starting from core.reaction.Complex_Degradation_Reaction._set_proteasomal_degration()

            reactions_to_add = []
            self.complex_degradation_reactions = complex_degradation_reactions
            for r in self.complex_degradation_reactions:
                if len(r.genes)>0:
                    rxn_mach = parse_complex.eval_complex(r.gene_reaction_rule)
                    for rm in rxn_mach:
                        if type(rm) != list:
                            rm = [rm]
                        else:
                            rm = sorted(rm)
                        present = self.complex_df[(self.complex_df.machinery == ';'.join(rm)) & (self.complex_df.compartment == func.get_reaction_compartment(r)) \
                           & (self.complex_df.reaction_id == parse_complex_degradation_reaction_id(r.id))]
                        if present.shape[0] == 0:
                            reactions_to_add.append(r)
            # Option 1: degrade rRNA with ribosomal degradation
            if  len(reactions_to_add) != 2 or not np.all([r._ribosomal_degradation for r in reactions_to_add]):
                err = 'Internal: Expected proteasomal degradation of ribosomal complexes to be the only difference in '
                err += 'machinery of complex degradation reactions. If other missed ones (perhaps in very small model '
                err += 'scenarios, but seems unlikely), will have to account for iteratitively adding new machinery '
                err += 'degradation reactions'
                print(err)
                raise ValueError('See internal error message above')
        #      # Option 2: degrade proteins with ribosomal degradation, releasing rRNA as intact
        #     if  len(reactions_to_add) != 0:
        #         err = 'Internal: Expected no additional machinery'
        #         print(err)
        #         raise ValueError('See internal error message above')    

        # if want, in the future, can back-track and remove associated expression reactions with these
        self.additional_ko = list(set(self.additional_ko))
        # #started some code for this:
        # for hgnc_id in additional_ko:
        #     if len(self.id_protein_map[hgnc_id])>1:
        #         err = 'Internal: Have not accounted for scenario where knocked out complex has an associated gene'
        #         err = 'that was not explicitly knocked out but '
        #         raise ValueError(err)
        #     else:
        #         self._ko_protein_map[hgnc_id] = self.id_protein_map.pop(hgnc_id)
        #         # remove expression reactions...
        
    def get_keff(self):
        '''Calculate the keff of all enzymes'''
        # calculated beforeprotein minimization to get average of all proteins
        # get SASA and keff values for coupling
        print('Calculate enzyme k_effs')
        # retain knocked-out genes in keff calculations 
        # do not include non_machinery
        cplx_bool_map = {True: 'complex', False: 'monomer'}
        ko_bool_map = {True: 'ko', False: 'retain'}
        mach_col_map = {True: 'complex_id', False: 'machinery'}
        ko_map = {'complex': {'ko': self._ko_complex_id_metabolite_map, 'retain': self.complex_id_metabolite_map},
                      'monomer': {'ko': self._ko_id_protein_map, 'retain': self.id_protein_map}}

        self.complex_df['MW_kDa'] = float('nan')
        for i in tqdm(self.complex_df.index):
            cplx_bool, ko_bool, _compartment = self.complex_df.loc[i, ['is_complex', 'knock_out', 'compartment']]
            _mach = self.complex_df.loc[i, mach_col_map[cplx_bool]]

            if not cplx_bool:
                enzyme_to_couple = ko_map[cplx_bool_map[cplx_bool]][ko_bool_map[ko_bool]][_mach][_compartment]
            else:
                enzyme_to_couple = ko_map[cplx_bool_map[cplx_bool]][ko_bool_map[ko_bool]][_mach]
            self.complex_df.loc[i, 'MW_kDa'] = enzyme_to_couple.formula_weight/1000 

        self.complex_df['SASA'] = self.complex_df.MW_kDa.apply(lambda x: func.SASA(x))
        median_SASA = self.complex_df.SASA.median()
        self.complex_df['keff'] = self.complex_df['SASA'].apply(lambda x: x*(params.keff_median/median_SASA))

        if self.dummy_protein is not None:
            self.dummy_protein['protein_metabolite'].keff = func.SASA(self.dummy_protein['protein_metabolite'].formula_weight/1000)*(params.keff_median/median_SASA)
    
    def minimize_proteome(self):
        c_og = self.complex_df.copy()
        n_reactions_og = len(self.me_reactions) + len(self.complex_formation_reactions)
        if self.deg_args['complex_degradation']:
            n_reactions_og += len(self.complex_degradation_reactions)

        drop_index = list()
        reaction_multiple = self.complex_df[self.complex_df.creates_multiple_reactions].reaction_id.unique().tolist()
#         blocked_reactions = dict(zip(reaction_multiple, [False]*len(reaction_multiple)))
        for rm in reaction_multiple:
            df = self.complex_df[self.complex_df.reaction_id == rm]
            # deal w/ knockouts
            to_drop = list()
            
            check = False
            if df[df.knock_out].shape[0] > 0 and df[df.knock_out].shape[0] < df.shape[0]:
                check = True
                to_drop += df[df.knock_out.astype(bool)].index.tolist()
                df_ = df[~df.knock_out.astype(bool)]
            else:
                df_ = df
            # don't directly drop machinery in case they are used in multiple reactions and are minimal in 
            # another one of those reactions
            to_drop += df_[df_.MW_kDa != df_.MW_kDa.min()].index.tolist() 
            if df.shape[0] - len(to_drop) == 1:
                drop_index += to_drop
            elif df.machinery.unique().shape[0] == df.shape[0]: 
                if check:
                    raise ValueError('Make sure proceeding line of code is correct for knock-out situation, currently has not been tested')
                # rare case where two different complexes have the same MW
                # instead of randomly choosing, to have consistent results, just choose the first option that appears
                drop_index += df_.index.tolist()[1:]
            else:
                raise ValueError('Something went wrong in selecting a complex by lowest molecular weight')

        self.complex_df.drop(index = drop_index, inplace = True)
#         self.complex_df['blocked_minimalproteome_complex'] = self.complex_df.reaction_id.map(blocked_reactions).tolist()

        # get rid of redundant complexes
        complexes_to_drop = sorted(set(c_og[c_og.is_complex & \
                                            ~c_og.knock_out.astype(bool)].complex_id).difference(self.complex_df.complex_id))#sorted(set(self.complex_id_metabolite_map.keys()).difference(self.complex_df.complex_id))
        complexes_to_drop_id = func.flatten_list([self.complex_reactions_map[c_id] for c_id in complexes_to_drop])
        self.complex_formation_reactions = [r for r in self.complex_formation_reactions if r.id not in complexes_to_drop_id]

        # backtrack and remove all protein expression and complex formation reactions of dropped enzymes
        if self.deg_args['complex_degradation']:
            self.complex_degradation_reactions = [r for r in self.complex_degradation_reactions if r.id not in complexes_to_drop_id]

        for c_id in complexes_to_drop:
            del self.complex_reactions_map[c_id]
            del self.complex_id_metabolite_map[c_id]

        # get the active monomer enzymes that are dropped
        prot_to_drop = set(c_og[~c_og.is_complex.astype(bool) & \
                                ~c_og.knock_out.astype(bool)].machinery).difference(\
                        self.complex_df[~self.complex_df.is_complex.astype(bool)].machinery)
        # include the inactive monomers that were components of dropped complexes, and are not used in other complexes or as other monomers
        all_complex_mach = func.flatten_list([cm.split(';') for cm in c_og[c_og.is_complex == True].machinery.tolist()]) # former complex machinery
        remaining_complex_mach = func.flatten_list([cm.split(';') for cm in self.complex_df.machinery.tolist()]) # tests against current complexes and monomers
        dropped_mach = (set(all_complex_mach).difference(remaining_complex_mach)) # machinery dropped bc of dropping complexes
        prot_to_drop = prot_to_drop.union(dropped_mach)
        reactions_to_remove = []
        id_protein_map = self.id_protein_map.copy()
        for hgnc_id in prot_to_drop:
            complex_machinery = [i.split(';') for i in self.complex_df[(self.complex_df.is_complex)].machinery.tolist()]
            complex_machinery = sorted(set([item for sublist in complex_machinery for item in sublist]))
            if hgnc_id not in complex_machinery:
                reactions_to_remove += [r.id for r in self.id_reactions_map[hgnc_id]]
                del self.id_reactions_map[hgnc_id] 
                del self.id_protein_map[hgnc_id]
            else:
                for comp in id_protein_map[hgnc_id].keys():
                    complex_machinery = [i.split(';') for i in self.complex_df[(self.complex_df.is_complex) & (self.complex_df.compartment == comp)].machinery.tolist()]
                    complex_machinery = sorted(set([item for sublist in complex_machinery for item in sublist]))

                    if hgnc_id not in complex_machinery:
                        rr = [r.id for r in self.id_reactions_map[hgnc_id] if len(r.compartments.intersection([comp])) > 0]
                        reactions_to_remove += rr
                        self.id_reactions_map[hgnc_id] = [r for r in self.id_reactions_map[hgnc_id] if r.id not in rr]
                        self.id_protein_map[hgnc_id] = {k:v for k,v in self.id_protein_map[hgnc_id].items() if k != comp}
        self.me_reactions = [r for r in self.me_reactions if r.id not in reactions_to_remove]
        n_reactions = len(self.me_reactions) + len(self.complex_formation_reactions)
        if self.deg_args['complex_degradation']:
            n_reactions += len(self.complex_degradation_reactions)

        print('A total of {} reactions were dropped when forming a minimal proteome'.format(n_reactions_og - n_reactions))

    def add_metabolic_machinery(self):
        # deal with metabolic reactions first
        print('Add machinery to metabolic module reactions')
        self._check_catalysis_coefficient = {}
        metabolic_reactions = [r.id for r in self.m_model.reactions]
        reaction_counter = dict(zip(sorted(set(metabolic_reactions)), [0]*len(metabolic_reactions))) 
        final_reactions = []

        if self.check_all and self.complex_df[(self.complex_df.category == 'metabolic_reaction') & (self.complex_df.knock_out)].reaction_id.value_counts().unique() != np.array([1]):
            raise ValueError('Internal: Something went wrong in formatting complex df for knock out')

        for i in tqdm(self.complex_df[self.complex_df.category == 'metabolic_reaction'].index):
            reaction_id = self.complex_df.loc[i, 'reaction_id'] # original reaction id
            r = to_metabolic_reaction(reaction = self.m_model.reactions.get_by_id(reaction_id))

            ko = self.complex_df.loc[i, 'knock_out']
            if not ko:
                if not self.complex_df.loc[i, 'is_complex']:
                    enzyme_to_couple = self.id_protein_map[self.complex_df.loc[i, 'machinery']][self.complex_df.loc[i, 'compartment']]
                    
                    # back track assign synthesis attribute to monomeric enzymes
                    srs = [sr for sr in list(enzyme_to_couple.reactions)if enzyme_to_couple in sr.products and \
                           not isinstance(sr, Protein_Degradation_Reaction)]
                    if len(srs) != 1:
                        raise ValueError(enzyme_to_couple.id + ' has an incorrect number of associated synthesis reactions')
                    srs[0].synthesis, srs[0].synthesis_type = True, 'protein'
                else:
                    enzyme_to_couple = self.complex_id_metabolite_map[self.complex_df.loc[i, 'complex_id']]
                    enzyme_to_couple.get_k_deg()
                    if self.check_all and len([1 for r in list(enzyme_to_couple.reactions) if \
                                               (hasattr(r, 'synthesis') and r.synthesis and \
                                               enzyme_to_couple in r.products)]) != 1:
                        raise ValueError(enzyme_to_couple.id + ' has an incorrect number of associated synthesis reactions')
                enzyme_to_couple.keff = self.complex_df.loc[i, 'keff']

                # add machinery to substrate side
                c3 = (params.mu + enzyme_to_couple.k_deg)/enzyme_to_couple.keff

                if self.check_all:
                    if not enzyme_to_couple.enzyme:
                        if enzyme_to_couple.id in self._check_catalysis_coefficient:
                            raise ValueError('Enzyme exists but is not classified as one')
                        self._check_catalysis_coefficient[enzyme_to_couple.id] = [c3]
                    else:
                        self._check_catalysis_coefficient[enzyme_to_couple.id] += [c3]

                    if c3.subs(params.mu, 1) <= 0:
                        raise ValueError('The catalysis coupling constraint is negative for ' + enzyme_to_couple.id)
                enzyme_to_couple.couple(type = 'catalysis', value = -c3)

                if not r.reversibility:
                    r.couple(metabolites = enzyme_to_couple, types = 'catalysis')
                    reactions = [r]
                else: # add a forward and reverse reaction for reversible reactions
                    r_f,r_r = r.copy(), r.copy()
                    r_f.lower_bound, r_r.lower_bound, r_r.upper_bound = 0,0, abs(r.lower_bound)
                    r_r.add_metabolites({metab: -coeff for metab, coeff in r_r.metabolites.items()}, combine = False)

                    r_f.couple(metabolites = enzyme_to_couple, types = 'catalysis')
                    r_r.couple(metabolites = enzyme_to_couple, types = 'catalysis')


                    r_f.id, r_r.id = r_f.id + '_F', r_r.id + '_R'
                    reactions = [r_f, r_r]
            else: # block metabolic reaction for knocked out genes
                r.lower_bound, r.upper_bound = 0, 0
                reactions = [r]

            # if multiple of the same reaction with different machinery due to OR GPR, add a different id for each
            if self.complex_df.loc[i, 'creates_multiple_reactions']:
                if len(reactions) > 1:
                    for j in range(len(reactions)):
                        r_ = reactions[j]
                        r_.id = r_.id + '_' + str(reaction_counter[reaction_id])
                        reactions[j] = r_
                if reaction_counter[reaction_id] == 0: # tracking that all metabolic reactions are added
                    metabolic_reactions.remove(reaction_id)
                reaction_counter[reaction_id] += 1
            else:
                metabolic_reactions.remove(reaction_id) # tracking that all metabolic reactions are added
            final_reactions += reactions

        # dummy protein for orphan reactions (see deorphan)
        if sorted(metabolic_reactions) != sorted([r.id for r in self.m_model.reactions if len(r.genes) == 0]):
            raise ValueError('Not all metabolic reactions that require machinery have been accounted for')
        if self.dummy_protein is None:
            self.orphan = [to_metabolic_reaction(r) for r in self.m_model.reactions if len(r.genes) == 0]
            final_reactions += self.orphan
            self.deorphaned = None
        self.final_reactions = final_reactions
    def add_expression_machinery(self):
        # filter out metabolic reactions
        backup = self.complex_df.copy()
        self.complex_df = self.complex_df[self.complex_df.category == 'expression_reaction']
        self.complex_df.reset_index(inplace = True, drop = True)

        if not self.deg_args['complex_degradation']:
            expression_reactions = self.me_reactions
        else:
            expression_reactions = self.me_reactions + [r for r in self.complex_degradation_reactions if not r._ribosomal_degradation]
            # filter out ribosomal_degradation reactions
        expression_reactions = [r for r in expression_reactions if len(r.genes)>0]
        reaction_counter = dict(zip(sorted(set([r.id for r in expression_reactions])), [0]*len(expression_reactions))) 

        print('Add machinery to expression module reactions')
        for rxn in tqdm(expression_reactions):
            if type(rxn) != Complex_Degradation_Reaction:
                reaction_id_short = func.parse_me_reaction_id(rxn.id) # abbreviated version
            else:
                reaction_id_short = parse_complex_degradation_reaction_id(rxn.id)

            reaction_id = rxn.id # original reaction id
            idx = self.complex_df[self.complex_df.reaction_id == reaction_id_short].index.tolist()
            for i in idx:
                r = rxn.copy()
#                 r._metabolites = rxn.metabolites
                if not self.complex_df.loc[i, 'is_complex']:
                    enzyme_to_couple = self.id_protein_map[self.complex_df.loc[i, 'machinery']][self.complex_df.loc[i, 'compartment']]
                    # back track assign synthesis attribute to monomeric enzymes
                    srs = [sr for sr in list(enzyme_to_couple.reactions)if enzyme_to_couple in sr.products and \
                           not isinstance(sr, Protein_Degradation_Reaction)]
                    if len(srs) != 1:
                        raise ValueError(enzyme_to_couple.id + ' has an incorrect number of associated synthesis reactions')
                    srs[0].synthesis, srs[0].synthesis_type = True, 'protein'
                else:
                    enzyme_to_couple = self.complex_id_metabolite_map[self.complex_df.loc[i, 'complex_id']]
                    enzyme_to_couple.get_k_deg()
                    if self.check_all and len([1 for r in list(enzyme_to_couple.reactions) if \
                                               (hasattr(r, 'synthesis') and r.synthesis and \
                                               enzyme_to_couple in r.products)]) != 1:
                        raise ValueError(enzyme_to_couple.id + ' has an incorrect number of associated synthesis reactions')
                enzyme_to_couple.keff = self.complex_df.loc[i, 'keff']

                # add machinery to substrate side
                c3 = (params.mu + enzyme_to_couple.k_deg)/enzyme_to_couple.keff

                if self.check_all:
                    if not enzyme_to_couple.enzyme:
                        if enzyme_to_couple.id in self._check_catalysis_coefficient:
                            raise ValueError('Enzyme exists but is not classified as one')
                        self._check_catalysis_coefficient[enzyme_to_couple.id] = [c3]
                    else:
                        self._check_catalysis_coefficient[enzyme_to_couple.id] += [c3]

                    if c3.subs(params.mu, 1) <= 0:
                        raise ValueError('The catalysis coupling constraint is negative for ' + enzyme_to_couple.id)
                enzyme_to_couple.couple(type = 'catalysis', value = -c3)

                if not r.reversibility:
                    r.couple(metabolites = enzyme_to_couple, types = 'catalysis')
                    reactions = [r]
                else: # add a forward and reverse reaction for reversible reactions
                    r_f,r_r = r.copy(), r.copy()
#                     r_f._metabolites, r_r._metabolites = rxn.metabolites, rxn.metabolites
                    r_f.lower_bound, r_r.lower_bound, r_r.upper_bound = 0,0, abs(r.lower_bound)
                    r_r.add_metabolites({metab: -coeff for metab, coeff in r_r.metabolites.items()}, combine = False)

                    r_f.couple(metabolites = enzyme_to_couple, types = 'catalysis')
                    r_r.couple(metabolites = enzyme_to_couple, types = 'catalysis')

                    r_f.id, r_r.id = r_f.id + '_F', r_r.id + '_R'
                    reactions = [r_f, r_r]

                # if multiple of the same reaction with different machinery due to OR GPR, add a different id for each
                if self.complex_df.loc[i, 'creates_multiple_reactions']:
                    if len(reactions) > 1:
                        for j in range(len(reactions)):
                            r_ = reactions[j]
                            r_.id = r_.id + '_' + str(reaction_counter[reaction_id])
                            reactions[j] = r_
                    reaction_counter[reaction_id] += 1
                self.final_reactions += reactions

        if self.deg_args['complex_degradation']:
            # hard-coded for ribosomal degradation
            enzyme_to_couple = [self.complex_id_metabolite_map[self.complex_df[self.complex_df.reaction_id == 'PROTEASOMAL_DEGRADATIONc'].complex_id.tolist()[0]]]
            enzyme_to_couple.append(self.complex_id_metabolite_map[self.complex_df[self.complex_df.reaction_id == '5s_rRNA_DEGRADATIONc'].complex_id.tolist()[0]])
            ribosomal_degradation_reactions = [r for r in self.complex_degradation_reactions if r._ribosomal_degradation]

            for rxn in ribosomal_degradation_reactions:
                rxn.couple(metabolites = enzyme_to_couple, types = ['catalysis', 'catalysis'])
                self.final_reactions.append(rxn)

        if self.dummy_protein is None:
            me_orphans = [r for r in self.me_reactions if len(r.genes) == 0]
            if self.deg_args['complex_degradation']:
                me_orphanss += [r for r in self.complex_degradation_reactions if len(r.genes) == 0]
            
            me_orphans += self.complex_formation_reactions     
            self.orphan += me_orphans
            self.final_reactions += me_orphans
            del me_orphans

        self.complex_df = backup.copy()
        del backup

        if self.check_all:
            for k,v in self._check_catalysis_coefficient.items():
                if len(set(v)) != 1:
                    raise ValueError(k + ' received multiple coupling coefficients for different reactions')
        del self._check_catalysis_coefficient
    
    def deorphan(self, exclude = None):
        '''Couples dummy protein to reactions that don't have specified genes ("de-orphaning")

        Parameters
        ----------
        exclude: list, default None
            a list of M_model reaction ids to exclude from coupling to dummy protein (if None, defaults to 
            non-exchange/demand metabolic model reactions and transport reactions; recommended default)

        Returns
        ----------
        deorphaned: list
            a list of ME_Model reaction IDs for reactions that were de-orphaned
        self.orphan: list
            a list of ME_Model reaction IDs for reactions there were not de-orphaned despite having 0 specified genes 

        '''
        

        if self.dummy_protein is not None:
            print('Deorphan enzymeless reactions')
            enzymeless_reactions = [to_metabolic_reaction(r) for r in self.m_model.reactions if len(r.genes) == 0]
            enzymeless_reactions_map = {r.cobra_id: r for r in enzymeless_reactions}
            enzymeless_reactions += [r for r in self.me_reactions if len(r.genes) == 0] + self.complex_formation_reactions
            if self.deg_args['complex_degradation']:
                enzymeless_reactions += [r for r in self.complex_degradation_reactions if len(r.genes) == 0]
            boundary_ids = [r.id for r in self.m_model.exchanges + self.m_model.demands]

            if len(set([r.id for r in enzymeless_reactions]).intersection([r.id for r in self.final_reactions])) > 0:
                raise ValueError('Incorrect parsing of reaction lists for dummy protein')
            if exclude is None:
                # metabolic module enzymes to exclude from deorphaning - boundary reactions 
                self.orphan = [r for r in enzymeless_reactions_map.values() if hasattr(r, 'cobra_id') and r.cobra_id in boundary_ids]
                _orphan = list()
                for r in self.orphan: # secondary exchange reactions
                    if len(r.metabolites) > 1 or list(r.metabolites)[0].compartment != 'b':
                        raise ValueError('Incorrectly formatted exchange reaction: ' + r.id + '. Must follow Recon2.2 format.')

                    assoc_rxn = assoc_rxn = [r_.id for r_ in list(list(self.m_model.reactions.get_by_id(r.id).metabolites)[0].reactions)]
                    assoc_rxn.remove(r.cobra_id)

                    if len(assoc_rxn) > 0:
                        for r_id in assoc_rxn: # id the second exchange reaction (Recon2.2 format)
                            r_ = self.m_model.reactions.get_by_id(r_id)
                            cond1 = (sorted(r_.compartments) == ['b', 'e'])
                            cond2 = (len(set(['_'.join(m.id.split('_')[:-1]) for m in list(r_.metabolites)])) == 1)
                            cond3 = (len(r_.genes) == 0)
                            if cond1 and cond2 and cond3 :
                                _orphan.append(enzymeless_reactions_map[r_.id])
                self.orphan += _orphan
                del _orphan
            else:
                raise ValueError('Exclude argument is deprecated')
                m_ids = [r.id for r in self.m_model.reactions]
                for r_id in exclude:
                    if r_id not in m_ids:
                        raise ValueError('The list of metabolic reactions to exclude from dummy catalysis must be in the metabolic model reaction list')
                    if len(self.m_model.reactions.get_by_id(r_id).genes)>0:
                        raise ValueError('The list of metabolic reactions to exclude from dummy catalysis must not have an associated GPR')

                self.orphan = [to_metabolic_reaction(r) for r in exclude]

            # expression module enzymes to exclude
            expression_rids = ['CYTOSOLIC_PROTEIN_FOLDING', 'IMPORTtn', 
                  'RIBOSOME_COMPLEX_DISSOCIATIONc', 'UNFOLDr',
                  'POLYUBIQUITIN_MOIETY_EXPORTtn', 'COMPLEX_FORMATION'] 
            for r in enzymeless_reactions:
                for expr_rid in expression_rids:
                    if r.id.__contains__(expr_rid):
                        self.orphan.append(r)
                        break

            # do not deorphan transport reactions for small molecules (can passively diffuse)
            transport = [r for r in enzymeless_reactions if len(r.compartments) > 1 and r not in self.orphan]
            remove_idx = list()
            for i in range(len(transport)):
                r = transport[i]
                tm = dict()
                mc = dict()
                counter = 0
                for m in r.metabolites:
                    if m.formula_weight <= params.membrane_diffusion_limit: # all metabolites within diffusion limit
                        counter += 1
                    m_id = '_'.join(m.id.split('_')[:-1]) # atleast one metabolite is transported across compartments
                    if m_id not in tm:
                        tm[m_id] = 1
                    else:
                        tm[m_id] += 1
                        mc[m_id] = m.charge

                uncharged = True
                for m_id,v in tm.items():
                    if v >= 2 and mc[m_id] != 0:
                        uncharged = False
                        break
                # uncharged, all metabolites that are transported are < 504 Da, and atleast one metabolite is transported 
                if max(list(tm.values())) >= 2 and counter == len(r.metabolites) and uncharged:
                    self.orphan.append(r)            

            deorphan = [r for r in enzymeless_reactions if r not in self.orphan]
            self.deorphaned = list()

            if len(deorphan) > 0:
                c3 = (params.mu + self.dummy_protein['protein_metabolite'].k_deg)/self.dummy_protein['protein_metabolite'].keff
                self.dummy_protein['protein_metabolite'].couple(type = 'catalysis', value = -c3)

                for r in deorphan:
                    if not r.reversibility:
                        r.couple(metabolites = self.dummy_protein['protein_metabolite'], types = 'catalysis')
                        reactions = [r]
                    else: # add a forward and reverse reaction for reversible reactions
                        r_f,r_r = r.copy(), r.copy()
                        r_f.lower_bound, r_r.lower_bound, r_r.upper_bound = 0,0, abs(r.lower_bound)
                        r_r.add_metabolites({metab: -coeff for metab, coeff in r_r.metabolites.items()}, combine = False)

                        r_f.couple(metabolites = self.dummy_protein['protein_metabolite'], types = 'catalysis')
                        r_r.couple(metabolites = self.dummy_protein['protein_metabolite'], types = 'catalysis')
                        r_f.id, r_r.id = r_f.id + '_F', r_r.id + '_R'
                        reactions = [r_f, r_r]
                    self.deorphaned += reactions
            self.final_reactions += self.orphan + self.deorphaned
            self.orphan = [r.id for r in self.orphan + biomass.biomass_reactions + [biomass.upb_reaction]]
            self.deorphaned = [r.id for r in self.deorphaned]
            
    def incorporate_protein_degradation(self):
        '''Removes degradation reactions of inactive monomers and couples protein degradation to catalysis, depending on 
        deg_args input'''

        if self.check_all and self.deg_args['complex_degradation']:
                for r in self.complex_degradation_reactions:
                    r._update_enzymes() # updates rxn ._enzymes attribute to include all macromolecules involved in reaction catalysis
                    if not (len(r._enzymes) > 0):
                        raise ValueError(r.id + ': this Complex_Degradation_Reaction is not associated with an active enzyme')

        if not self.deg_args['nonenzyme_degradation']: # remove degradation reactions of nonenzymes (degraded in complex)
            reactions_to_remove = list()
            pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]
            for r in pdr:
                r._update_enzymes()
                if not(len(r._enzymes) > 0):
                    reactions_to_remove.append(r.id)
            print('{} of {} protein degradation reactions will be removed because they are not associated with an active enzyme'.format(len(reactions_to_remove), len(pdr)))

            if self.check_all and len(set(reactions_to_remove).difference([r.id for r in self.final_reactions]))>0:
                raise ValueError('Untracked protein degradation reactions (not in final reactions list)')

            self.final_reactions = [r for r in self.final_reactions if r.id not in reactions_to_remove]
        if self.deg_args['couple']:
            print('Couple enzyme degradation to catalysis')

            pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]
            dr_map = dict()
            for r in pdr + self.complex_degradation_reactions:
                r._update_enzymes()
                dr_map[r.id] = r
            catalysis_reactions = [r for r in self.final_reactions if r.coupled_metabolites != dict() \
                                   and 'catalysis' in r.coupled_metabolites.values() and \
                                   func.get_reaction_compartment(r) != 'e' and not (r.lower_bound == r.upper_bound == 0)]


            # TEMPORARY: don't couple ribosomal degradation for now - change in Expressed_Gene._check_macromolecules
            catalysis_reactions = [r for r in catalysis_reactions if not('mrna_formation') in r.coupled_metabolites.values()]

            for r in tqdm(catalysis_reactions):
                enzymes = [m for m,t in r.coupled_metabolites.items() if t == 'catalysis']


                deg_reactions = list()
                deg_proxies = list()
                for e in enzymes: # in case multiple catalysis proteins (ribosomal degradatio)
                    deg_reactions_ = [r_id for r_id in e._degradation_reactions if (dr_map[r_id].sink) and (e in dr_map[r_id]._enzymes)]
                    if len(deg_reactions_) == 0:
                        raise ValueError('No degradation reactions associated with catalyzing enzyme')
                    elif len(deg_reactions_) > 1:
                        raise ValueError('More than 1 degradation reaction associated with catalyzing enzyme')
                    deg_reactions += deg_reactions_
                    dp = e.make_proxy()
                    dp.couple(value = -e.k_deg/e.keff)
                    deg_proxies.append(dp)



                if len(deg_reactions) > 1 and not (r.id in dr_map or dr_map[r.id]._ribosomal_degradation):
                    raise ValueError('More than 1 degradation reaction associated with the catalysis reaction')

                deg_reactions = [r_ for r_ in self.final_reactions if r_.id in deg_reactions]
                for dr, dp in list(zip(deg_reactions, deg_proxies)):
                    # keeping track of whether the enzyme degradation reaction already has a proxy metabolite for 
                    # coupling (occurs in scenarios where an enzyme catalyzes multiple reactions)
                    if not dr._protein_deg_proxy: 
                        dr._add_protein_deg_proxy(dp)
                    else:
                        dp = dr.protein_deg_proxy

                    if (self.check_all) and ('enzyme_degradation' in r.coupled_metabolites.values()) and (not dr_map[r.id]._ribosomal_degradation): 
                        raise ValueError('This reaction already is coupled to degradation')
                    r.couple(metabolites = dp, types = 'enzyme_degradation')
                    #.couple works in scenarios where r == dr because .couple uses .add_metabolites(combine = True)

    def build_me_model(self, model_id = 'HUMAN_ME_MODEL'):
        print('Add biomass component to reactions')

#         for r in self.final_reactions: # correct wrong metabolite tracking
#             if r.coupled_metabolites is not None:
#                 mmap = {m.id: m for m in r.metabolites}
#                 cm = dict()
#                 for md,type_ in r.coupled_metabolites.items():
#                     cm[mmap[md.id]] = type_
#                 r.coupled_metabolites = cm 

        for r in self.final_reactions:
            biomass.add_biomass_change(r)

        br = [r.copy() for r in biomass.biomass_reactions]
        #         br.append(self.pb_reaction) 
        if self.dummy_protein is not None:
            br.append(biomass.upb_reaction.copy()) 

        if len([r for r in self.final_reactions if not isinstance(r, core.reaction.ME_Reaction)])>0:
            raise ValueError('Internal: Reactions not of type ME_Reaction are included in the model')
        self.final_reactions += br

        print('Generate ME-Model')
        me_model = ME_Model(m_model = self.m_model, id_or_model = model_id, n_cores = self.n_cores)
        
        # note, at end of .add_reactions() method, we reassign .coupled_metabolites attribute
        # running .add_metabolic_reactions() code outside of ME_Builder object doesn't create disagreement
        # between r.metabolites and r.coupled_metabolites, but running the method on the object does
        me_model.add_reactions(self.final_reactions)
        me_model.reaction_types['orphan'] = self.orphan
        me_model.reaction_types['deorphaned'] = self.deorphaned

        me_model.check(orphan = self.orphan, 
                       knock_out = self.knock_out, _additional_ko = self.additional_ko)
        me_model._generate_expressed_genes()
        
#         del self.pb_reaction
#         del self.ub_args
        del self.me_reactions
        del self.final_reactions
        del self.complex_formation_reactions
        del self.complex_degradation_reactions
        del self.m_model
        del self.orphan
        del self.deorphaned

        return me_model


In [5]:
def build_me(minimal_proteome = True, compress_mrna = True, dummy_protein = True,
             deg_args = {'couple': True, 'reversible_complex_formation': False, 'nonenzyme_degradation': False, 
                          'complex_degradation': True}, check_all = True, non_machinery = dict(),
             model_id = 'HUMAN_ME_MODEL', knock_out = None):
    '''Generates a human ME_model. 
    
    Parameters
    ----------
    minimal_proteome: bool
        For reactions with OR in the GPR, the builder by default (False) generates a 
        separate reaction for each protein complex (False). If True, builder instead will create one reaction, 
        choosing the protein complex with the lowest molecular weight to catalyze the reaction. If a reaction
        has multiple enzyme options with the same molecular weight, will randomly choose one. Will not consider
            a complex that contains a knocked out gene. 
    compress_mrna: bool
        If true, will merge the 3 linear mrna reactions--transcription, processing, and nuclear export--for each
        gene into a single reaction
    dummy_protein: bool [True]
        whether to add a representative dummy protein to catalyze orphan reactions 
    deg_args: dict
        A number of options related to protein and complex degradation. Becomes important in slow growth conditions.
        Note the default values focus on coupling fluxes and degrading the specific enzymes associated with 
        each reaction. 

        Key value pairs:
            "couple": bool
                Whether to explicitly couple enzyme degradation reactions to metabolic catalysis. Becomes 
                particularly important in slow growth conditions.
            "reversible_complex_formation": bool
                Whether reactions to form complexes are reversible (<->) or not (-->). Setting to True may make
                model more efficient (reuse of proteins involved in catalysis of multiple reactions in same compartment)
            "nonenzyme_degradation": bool
                Whether to retain degradation reactions (associated with the build_protein_expression script) for
                proteins that form complexes rather than become monomeric enzymes; i.e., all individual complex 
                subunits have their own protein degradation reaction. Note that even if set to False, 
                protein intermediates associated with the monomeric enzyme that had degradation rections are retained.
                Regardless of this parameter, only the specific enzymatic degradation reaction associated with the 
                catalysis reaction will be coupled. Independent of complex_degradation and 
                reversible_complex_formation arguments.
            "complex_degration": bool
                Whether to generate degradation reactions for whole complexes in addition to individual monomers
                (required for coupling)
    check_all: bool
        Whether to check that building is proceeding correctly. Increases run time
    model_id: str
        id for the me model
    knock_out: list
        each element is a string representing a gene expressed in the model which should be knocked out
        *Note: you may want to knock-out during building if setting minimal_proteome = True and knocking out a 
        gene that participates in a OR GPR rule(in case it is the one that is selected); otherwise 
        me_model.knock_out() method should suffice
    non_machinery: dictionary
        keys are HGNC IDs, values are a list of strings, each element of which represents a compartment
        within the metabolic model for the gene to be expressed
    '''
    
    start = time.time()
    builder = me_builder(compress_mrna = compress_mrna, 
                         dummy_protein = dummy_protein, deg_args = deg_args, check_all = check_all, 
                        knock_out = knock_out, non_machinery = non_machinery)
    builder.express_metabolic_enzymes()
    builder.express_expression_enzymes()
    builder.express_dummy_protein()
    builder.get_complex_info()
    builder.generate_complex_reactions()
    builder.get_keff()
    if minimal_proteome:
        builder.minimize_proteome()
    builder.add_metabolic_machinery()
    builder.add_expression_machinery()
    builder.deorphan()
    builder.incorporate_protein_degradation()
    me_model = builder.build_me_model(model_id = model_id)

    end = time.time()
    print('Time to build: {} minutes'.format((end-start)/60))


    return me_model, builder


In [6]:
gc.collect()

0

In [21]:
n_cores = 15
minimal_proteome = True
compress_mrna = True
dummy_protein = True
non_machinery = {'HGNC:4556':['m', 'c'], #metabolic machinery only, m is an enzymatic compartment, c is not
                'HGNC:9251': ['l'], # expr + metab machinery, l is m machinery and expression machinery compartment
                 'HGNC:30076': ['n', 'm'], # eexpr machinery only, n is an enzymatic compartment, m is not
                'HGNC:32043': ['e', 'n']} # pure non-machinery
deg_args = {'reversible_complex_formation': True, 
                       'couple': True,
                       'nonenzyme_degradation': False, 
                       'complex_degradation': True}
model_id = 'HUMAN_ME_MODEL'

builder = me_builder(n_cores = n_cores, compress_mrna = compress_mrna,
                         dummy_protein = dummy_protein, deg_args = deg_args, non_machinery = non_machinery)
# builder.express_metabolic_enzymes()
# builder.express_expression_enzymes()
# builder.express_dummy_protein()
# builder.get_complex_info()
# builder.generate_complex_reactions()
# builder.get_keff()
# if minimal_proteome:
#     builder.minimize_proteome()
# builder.add_metabolic_machinery()
# builder.add_expression_machinery()
# builder.deorphan()

# self = copy.deepcopy(builder)

Generate ubiquitin reactions for proteasomal degradation
Generate ribosome


In [8]:
builder.non_machinery

{'HGNC:4556': ['c'], 'HGNC:30076': ['m'], 'HGNC:32043': ['e', 'n']}

In [9]:
test = list()
for v in builder.id_protein_map.values():
    for compartment, metab in v.items():
        if metab.non_machinery:
            test.append(metab)
test

[<Protein HGNC:30076_folded_pre_protein_m at 0x7fc298234320>,
 <Protein HGNC:32043_folded_protein_n at 0x7fc272471630>,
 <Protein HGNC:32043_folded_protein_e at 0x7fc272471668>]

In [31]:
builder.non_machinery

{'HGNC:4556': ['m', 'c'],
 'HGNC:9251': ['l'],
 'HGNC:30076': ['n', 'm'],
 'HGNC:32043': ['e', 'n']}

In [37]:
gene_reaction_map = func.create_gene_reaction_map(params.human_model.reactions)
for hgnc_id in self.non_machinery:
    if hgnc_id not in gene_reaction_map:
        gene_reaction_map[hgnc_id] = None

In [38]:
expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, 
                                                reactions = gene_reaction_map[hgnc_id],
                                              compress_mrna = self.compress_mrna, 
                                                ub_args = self.ub_args, nonmachinery_locations = nml)

In [40]:
protein_metabolites[0].non_machinery

True

In [43]:
expr_reactions[-5]._non_machinery

False

In [10]:
# get protein expression for all metabolic reactions
print('Generate protein expression reactions for metabolic enzymes and non-machinery')


self.loop_machinery = list(set(mach.metabolic_machinery + list(self.non_machinery)))

gene_reaction_map = func.create_gene_reaction_map(params.human_model.reactions)
for hgnc_id in self.non_machinery:
    if hgnc_id not in gene_reaction_map:
        gene_reaction_map[hgnc_id] = None


iterable = set(self.loop_machinery).difference(self.knock_out)
if not self._par:
    for hgnc_id in tqdm(iterable):
        nml = list()
        if hgnc_id in non_machinery:
            nml = non_machinery[hgnc_id]
        expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, 
                                                reactions = gene_reaction_map[hgnc_id],
                                              compress_mrna = self.compress_mrna, 
                                                ub_args = self.ub_args, nonmachinery_locations = nml)
        self.id_protein_map[hgnc_id] = {p.compartment: p for p in protein_metabolites} # store compartments and metabolite objects for each gene
        self.id_reactions_map[hgnc_id] = expr_reactions
        self.me_reactions += expr_reactions
else:
    pool = multiprocessing.Pool(processes = self.n_cores)
    try:
        n_iter = len(iterable)
        args = zip(iterable, [gene_reaction_map]*n_iter, [self.ub_args]*n_iter, [self.compress_mrna]*n_iter, [self.non_machinery]*n_iter)
        mm = pool.starmap(emm_par, args)
        pool.close()
        pool.join()
        gc.collect()
    except:
        pool.close()
        pool.join()
        gc.collect()
        raise ValueError('Parallelization failed')
    self.id_protein_map = dict(zip(iterable, [i[0] for i in mm]))
    expr_reactions = [i[1] for i in mm]
    self.id_reactions_map = dict(zip(iterable, expr_reactions))
    self.me_reactions += func.flatten_list(expr_reactions)
    del expr_reactions

for hgnc_id in self.knock_out:
    # None bc will add later for expression model specific to this
    expr_reactions, protein_metabolites = get_all_expression_reactions(hgnc_id, reactions = gene_reaction_map[hgnc_id], 
                                                                       compress_mrna = self.compress_mrna, 
                                                                      ub_args = self.ub_args)
    self._ko_id_protein_map[hgnc_id] = {p.compartment: p for p in protein_metabolites} # store compartments and metabolite objects for each gene

In [20]:
list(self.id_protein_map['HGNC:30076']['m'].reactions)[0]._non_machinery

False

In [22]:
if not self.deg_args['nonenzyme_degradation']: # remove degradation reactions of nonenzymes (degraded in complex)
    reactions_to_remove = list()
    pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]
    for r in pdr:
        r._update_enzymes()
        if not(len(r._enzymes) > 0):
            reactions_to_remove.append(r.id)
    print('{} of {} protein degradation reactions will be removed because they are not associated with an active enzyme'.format(len(reactions_to_remove), len(pdr)))

    if self.check_all and len(set(reactions_to_remove).difference([r.id for r in self.final_reactions]))>0:
        raise ValueError('Untracked protein degradation reactions (not in final reactions list)')

In [ ]:
pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]
for r in pdr:
    r._update_enzymes()
    if not(len(r._enzymes) > 0):
        reactions_to_remove.append(r.id)

In [24]:
pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]

In [34]:
test = [r for r in pdr if r.hgnc_id == 'HGNC:32043']

In [35]:
r = test[0]

In [39]:
r.__dict__.keys()

dict_keys(['_id', 'name', 'notes', '_annotation', '_gene_reaction_rule', 'subsystem', '_genes', '_metabolites', '_compartments', '_model', '_lower_bound', '_upper_bound', 'coupled_metabolites', '_protein_deg_proxy', 'ubiquitin_biogenesis', 'hgnc_id', 'synthesis', 'synthesis_type', 'sink', 'sink_type', 'ribosome_biogenesis', '_macromolecules', '_enzymes', '_ribosomal_degradation'])

In [40]:
r._macromolecules

[<Protein HGNC:32043_folded_protein_polyub_protein_c at 0x7f72076c8a20>,
 <Protein HGNC:32043_folded_protein_c at 0x7f72076c87f0>]

In [32]:
'HGNC:32043' in set([r.hgnc_id for r in self.me_reactions])

True

In [23]:
self.non_machinery

{'HGNC:4556': ['c'], 'HGNC:30076': ['m'], 'HGNC:32043': ['e', 'n']}

In [ ]:
if self.check_all and self.deg_args['complex_degradation']:
    for r in self.complex_degradation_reactions:
        r._update_enzymes() # updates rxn ._enzymes attribute to include all macromolecules involved in reaction catalysis
        if not (len(r._enzymes) > 0):
            raise ValueError(r.id + ': this Complex_Degradation_Reaction is not associated with an active enzyme')

if not self.deg_args['nonenzyme_degradation']: # remove degradation reactions of nonenzymes (degraded in complex)
    reactions_to_remove = list()
    pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]
    for r in pdr:
        r._update_enzymes()
        if not(len(r._enzymes) > 0):
            reactions_to_remove.append(r.id)
    print('{} of {} protein degradation reactions will be removed because they are not associated with an active enzyme'.format(len(reactions_to_remove), len(pdr)))

    if self.check_all and len(set(reactions_to_remove).difference([r.id for r in self.final_reactions]))>0:
        raise ValueError('Untracked protein degradation reactions (not in final reactions list)')

    self.final_reactions = [r for r in self.final_reactions if r.id not in reactions_to_remove]
if self.deg_args['couple']:
    print('Couple enzyme degradation to catalysis')

    pdr = [r for r in self.me_reactions if isinstance(r, Protein_Degradation_Reaction)]
    dr_map = dict()
    for r in pdr + self.complex_degradation_reactions:
        r._update_enzymes()
        dr_map[r.id] = r
    catalysis_reactions = [r for r in self.final_reactions if r.coupled_metabolites != dict() \
                           and 'catalysis' in r.coupled_metabolites.values() and \
                           func.get_reaction_compartment(r) != 'e' and not (r.lower_bound == r.upper_bound == 0)]


    # TEMPORARY: don't couple ribosomal degradation for now - change in Expressed_Gene._check_macromolecules
    catalysis_reactions = [r for r in catalysis_reactions if not('mrna_formation') in r.coupled_metabolites.values()]

    for r in tqdm(catalysis_reactions):
        enzymes = [m for m,t in r.coupled_metabolites.items() if t == 'catalysis']


        deg_reactions = list()
        deg_proxies = list()
        for e in enzymes: # in case multiple catalysis proteins (ribosomal degradatio)
            deg_reactions_ = [r_id for r_id in e._degradation_reactions if (dr_map[r_id].sink) and (e in dr_map[r_id]._enzymes)]
            if len(deg_reactions_) == 0:
                raise ValueError('No degradation reactions associated with catalyzing enzyme')
            elif len(deg_reactions_) > 1:
                raise ValueError('More than 1 degradation reaction associated with catalyzing enzyme')
            deg_reactions += deg_reactions_
            dp = e.make_proxy()
            dp.couple(value = -e.k_deg/e.keff)
            deg_proxies.append(dp)



        if len(deg_reactions) > 1 and not (r.id in dr_map or dr_map[r.id]._ribosomal_degradation):
            raise ValueError('More than 1 degradation reaction associated with the catalysis reaction')

        deg_reactions = [r_ for r_ in self.final_reactions if r_.id in deg_reactions]
        for dr, dp in list(zip(deg_reactions, deg_proxies)):
            # keeping track of whether the enzyme degradation reaction already has a proxy metabolite for 
            # coupling (occurs in scenarios where an enzyme catalyzes multiple reactions)
            if not dr._protein_deg_proxy: 
                dr._add_protein_deg_proxy(dp)
            else:
                dp = dr.protein_deg_proxy

            if (self.check_all) and ('enzyme_degradation' in r.coupled_metabolites.values()) and (not dr_map[r.id]._ribosomal_degradation): 
                raise ValueError('This reaction already is coupled to degradation')
            r.couple(metabolites = dp, types = 'enzyme_degradation')
            #.couple works in scenarios where r == dr because .couple uses .add_metabolites(combine = True)



In [ ]:
# calculated beforeprotein minimization to get average of all proteins
# get SASA and keff values for coupling
print('Calculate enzyme k_effs')
# retain knocked-out genes in keff calculations 
# do not include non_machinery
cplx_bool_map = {True: 'complex', False: 'monomer'}
ko_bool_map = {True: 'ko', False: 'retain'}
mach_col_map = {True: 'complex_id', False: 'machinery'}
ko_map = {'complex': {'ko': self._ko_complex_id_metabolite_map, 'retain': self.complex_id_metabolite_map},
              'monomer': {'ko': self._ko_id_protein_map, 'retain': self.id_protein_map}}

self.complex_df['MW_kDa'] = float('nan')
for i in tqdm(self.complex_df.index):
    cplx_bool, ko_bool, _compartment = self.complex_df.loc[i, ['is_complex', 'knock_out', 'compartment']]
    _mach = self.complex_df.loc[i, mach_col_map[cplx_bool]]

    if not cplx_bool:
        enzyme_to_couple = ko_map[cplx_bool_map[cplx_bool]][ko_bool_map[ko_bool]][_mach][_compartment]
    else:
        enzyme_to_couple = ko_map[cplx_bool_map[cplx_bool]][ko_bool_map[ko_bool]][_mach]
    self.complex_df.loc[i, 'MW_kDa'] = enzyme_to_couple.formula_weight/1000 

self.complex_df['SASA'] = self.complex_df.MW_kDa.apply(lambda x: func.SASA(x))
median_SASA = self.complex_df.SASA.median()
self.complex_df['keff'] = self.complex_df['SASA'].apply(lambda x: x*(params.keff_median/median_SASA))

if self.dummy_protein is not None:
    self.dummy_protein['protein_metabolite'].keff = func.SASA(self.dummy_protein['protein_metabolite'].formula_weight/1000)*(params.keff_median/median_SASA)

In [11]:
# calculated beforeprotein minimization to get average of all proteins
# get SASA and keff values for coupling
print('Calculate enzyme k_effs')
# retain knocked-out genes in keff calculations 
# do not include non_machinery
cplx_bool_map = {True: 'complex', False: 'monomer'}
ko_bool_map = {True: 'ko', False: 'retain'}
mach_col_map = {True: 'complex_id', False: 'machinery'}
ko_map = {'complex': {'ko': self._ko_complex_id_metabolite_map, 'retain': self.complex_id_metabolite_map},
              'monomer': {'ko': self._ko_id_protein_map, 'retain': self.id_protein_map}}

self.complex_df['MW_kDa'] = float('nan')

Calculate enzyme k_effs


In [16]:
allm = func.flatten_list([i.split(';') for i in self.complex_df.machinery.tolist()])

In [20]:
'HGNC:32043' in allm

False

In [ ]:
sel

In [7]:
# n_cores = 15
# minimal_proteome = True
# compress_mrna = True
# dummy_protein = True
# non_machinery = dict()
# deg_args = {'reversible_complex_formation': True, 
#                        'couple': True,
#                        'nonenzyme_degradation': False, 
#                        'complex_degradation': True}
# model_id = 'HUMAN_ME_MODEL'

# builder = me_builder(n_cores = n_cores, compress_mrna = compress_mrna,
#                          dummy_protein = dummy_protein, deg_args = deg_args, non_machinery = non_machinery)
# #builder.knock_out = ['HGNC:4249']#, 'HGNC:4139', 'HGNC:4801', 'HGNC:6936']

# builder.express_metabolic_enzymes()
# builder.express_expression_enzymes()
# builder.express_dummy_protein()
# builder.get_complex_info()
# builder.generate_complex_reactions()
# builder.get_keff()
# if minimal_proteome:
#     builder.minimize_proteome()
# builder.add_metabolic_machinery()
# builder.add_expression_machinery()
# builder.deorphan()
# builder.incorporate_protein_degradation()
# me_model = builder.build_me_model(model_id = model_id)

# lp_path = '/data2/hratch/human_me/other/test_lp/'
# counter = 4
# me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')